# 03 · Revisión interna y métricas

Para preparar la reunión con Martin y decidir qué prompts usar. Solo lee datos (salvo los veredictos de la sección 4).

0. Qué hay en cada corrida (versiones con resultados, anotación a ciegas o asistida)
1. Ranking de **todas** las versiones de prompt, por corrida
2. Métricas de los prompts en uso (v15, v14, v13, v12, v8) y de la **moda de los 3 mejores**, en A1, A2 y B
2d. Triaje por registro (estado y número de huesos): `triage_v2` vs `triage_v15`
2e. Recuadro del hueso: modelo vs recuadro dibujado a mano (IoU)
3. Explicación de las métricas
4. Persona vs modelos, foto por foto, con **veredicto** (*my error / model error / ask Martin*)
5. Registros posiblemente duplicados (mismo día y lugar)

**Cuidado con el sesgo:**
- A1 se anotó a ciegas: es la referencia para medir. A2 se anotó con los modelos a la vista: su acuerdo sale inflado. En B, la sección 0 dice cuántas fotos vienen de la asistida.
- **v15 solo se corrió en B** (19 nov): se compara con v12-v14 en B, sobre las mismas fotos. En A1 y A2 no aparece.
- El ranking y la moda se eligen con las mismas fotos de A1 con las que se evalúan: la cifra sale algo optimista. Hace falta un set de validación nuevo (notebook 02, sección 3).

| Notebook | Para qué |
|---|---|
| `01_descarga_y_modelos` | bajar registros de iNaturalist y correr el triaje y los 4 prompts |
| `02_anotacion` | anotar: asistida (elegir el mejor valor de los modelos), revisión con letra y recuadro, o a ciegas |
| `03_revision_y_metricas` | revisión interna: ranking de prompts, moda, métricas, persona vs modelos, duplicados |
| `04_revision_con_martin` | lo que se le muestra a Martin: qué hicimos, resultados, fotos, preguntas y el Excel |
| `05_entrega_excel` | generar y revisar el Excel de entrega |

La versión oficial de cada notebook está en GitHub (`notebooks/`, sin resultados). Para trabajar: abrirlo desde GitHub en Colab, *Archivo → Guardar una copia en Drive* y moverla a `MyDrive/sepia-inaturalist/notebooks/` (con la fecha en el nombre).

In [ ]:
# --- Preparacion (correr siempre primero) ---
# Monta Drive, clona o actualiza el repo y carga el codigo recien bajado.
from google.colab import drive, output, userdata
drive.mount('/content/drive')
output.enable_custom_widget_manager()          # interfaces (ipywidgets) en Colab

import os, sys, importlib
if not os.path.exists('/content/sepia-inaturalist'):     # sesion nueva: clonar el repo
    token = userdata.get('GITHUB_TOKEN')
    !git clone -q https://{token}@github.com/hugozeballos/sepia-inaturalist.git /content/sepia-inaturalist
%cd /content/sepia-inaturalist
!git pull -q
!git log -1 --oneline

sys.path.insert(0, '/content/sepia-inaturalist/src')
os.environ['PYTHONPATH'] = '/content/sepia-inaturalist/src'   # para los !python scripts/...
for m in [m for m in sys.modules if m == 'sepia' or m.startswith('sepia.')]:
    del sys.modules[m]                          # descartar la version vieja en memoria
sys.path_importer_cache.pop('/content/sepia-inaturalist/src', None)
importlib.invalidate_caches()
import sepia

# Prompts en uso (detalle en docs/prompt_versions.md). Cada corrida usa los que tiene:
# v15 solo esta en B; v8 solo en A1 y A2.
VERSIONS = ['photo_traits_v15', 'photo_traits_v14', 'photo_traits_v13', 'photo_traits_v12', 'photo_traits_v8']
TRIAGE_VERSIONS = ['triage_v2', 'triage_v15']
BONEBOX_VERSIONS = ['bonebox_v1', 'bonebox_v15']
ANNOTATOR = 'HZ'

In [ ]:
# Corridas (cada una en su carpeta de Drive). A1: anotacion a ciegas. A2: asistida. B: ver seccion 0.
cfg_a1 = sepia.load_config('configs/pilot_a1.yaml')
cfg_a2 = sepia.load_config('configs/a2_test.yaml')
cfg_b = sepia.load_config('configs/sector_b.yaml')
RUNS = [cfg_a1, cfg_a2, cfg_b]
run_name = lambda c: c.base_dir.rstrip('/').split('/')[-1]

In [ ]:
from sepia.disagree import (rank_versions, classification_metrics, plot_confusion, count_agreement,
                            DisagreementApp, verdict_summary, triage_agreement, MODE)
from sepia.photo_traits import load_photo_traits
from sepia.triage import load_triage
from sepia.bonebox import load_bonebox, compare_with_manual
from sepia.status import annotation_status

def available(c, versions=VERSIONS):
    """Versiones de photo_traits con resultados en esta corrida (en el orden de VERSIONS)."""
    return [v for v in versions if not load_photo_traits(c, v).empty]

def show_metrics(c, top3=None):
    """Conteo, metricas y matrices de los prompts en uso de una corrida (+ moda si estan los 3)."""
    vs = available(c)
    mode = top3 if top3 and all(v in vs for v in top3) else None
    print(f"===== {run_name(c)}: {', '.join(v.replace('photo_traits_', '') for v in vs)}"
          + (f" + moda de {', '.join(mode)}" if mode else " (sin moda: falta alguno de los 3 mejores)"))
    count_agreement(c, vs, mode_of=mode)
    summary, per_class, mats = classification_metrics(c, vs, save=False, mode_of=mode)
    for f in ('side', 'erosion', 'photo_quality'):
        plot_confusion(mats, f)
    return summary, per_class, mats

## 0. Qué hay en cada corrida

Versiones con resultados y avance de la anotación. `from_assisted` = individuos cuya anotación partió de la asistida (no es a ciegas: su acuerdo con los modelos sale inflado).

In [ ]:
import pandas as pd
rows = []
for c in RUNS:
    t = c.triage or {}
    rows.append({'run': run_name(c),
                 'photo_traits': ', '.join(v.replace('photo_traits_', '') for v in available(c)),
                 'triage': ', '.join(v for v in TRIAGE_VERSIONS
                                     if not load_triage(c, t.get('provider', 'gemini'), t.get('model'), v).empty),
                 'bonebox': ', '.join(v for v in BONEBOX_VERSIONS if not load_bonebox(c, v).empty)})
display(pd.DataFrame(rows).set_index('run'))
display(pd.concat([annotation_status(c) for c in RUNS], ignore_index=True))

## 1. Ranking de todas las versiones, por corrida

Evalúa todas las versiones con resultados en cada corrida sobre las mismas fotos anotadas, y las ordena por el promedio de acuerdo en conteo, lado, erosión y calidad. Ojo: `side_agreement` cuenta `unclear` como error; mirar también `side_agreement_when_decided`. Solo se comparan versiones **dentro** de una corrida (cada corrida tiene otras fotos).

In [ ]:
all_ranking = {}
for c in RUNS:
    print(f"\n################ {run_name(c)} ################")
    all_ranking[run_name(c)], _ = rank_versions(c, n_top=5, save=False)

## 2. Los prompts en uso y la moda de los 3 mejores

La **moda** toma, por foto y atributo, el valor que dan al menos 2 de los 3 mejores; si los 3 difieren, `unclear` (en el conteo: la foto se salta). Los 3 mejores se eligen entre los prompts en uso con el ranking de A1 (v15 no está en A1, así que no entra en la moda).

In [ ]:
ranking, _ = rank_versions(cfg_a1, available(cfg_a1), n_top=3, save=False)
TOP3 = list(ranking.index[:3])
print('3 mejores (A1):', TOP3)

### 2a. A1 (a ciegas)

In [ ]:
summary_a1, per_class_a1, mats_a1 = show_metrics(cfg_a1, TOP3)

### 2b. A2 (asistida, solo referencia)

In [ ]:
summary_a2, per_class_a2, mats_a2 = show_metrics(cfg_a2, TOP3)

### 2c. B (con v15)

Mismas fotos para todas las versiones. Revisar en la sección 0 cuántas anotaciones de B vienen de la asistida.

In [ ]:
summary_b, per_class_b, mats_b = show_metrics(cfg_b, TOP3)

### 2d. Triaje por registro: estado y número de huesos

Solo registros con **todas** sus fotos anotadas y no excluidos, y con resultado en todas las versiones comparadas. Persona: `n_cuttlebones` = letras distintas con estado hueso. `count_agreement` solo donde el modelo dijo `count_possible = true`; `count_possible_false` = registros que manda a conteo manual.

In [ ]:
triage_tables = {}
for c in RUNS:
    t = c.triage or {}
    vs = [v for v in TRIAGE_VERSIONS if not load_triage(c, t.get('provider', 'gemini'), t.get('model'), v).empty]
    if not vs:
        continue
    print(f"\n################ {run_name(c)} ################")
    triage_tables[run_name(c)] = triage_agreement(c, vs)
    if c is cfg_b:
        print('\n--- B sin los registros con anotacion asistida ---')
        triage_agreement(c, vs, exclude_assisted=True)

### 2e. Recuadro del hueso: modelo vs dibujado a mano

Fotos con un solo individuo anotado y con recuadro a mano. IoU: 0 = no se tocan, 1 = idénticos; ≥ 0,5 cuenta como acierto. `iou_yx` usa el orden pedido en el prompt.

In [ ]:
for c in RUNS:
    for v in BONEBOX_VERSIONS:
        cmp = compare_with_manual(c, v)
        if cmp.empty:
            continue
        iou = cmp['iou_yx'].dropna()
        print(f"{run_name(c)} · {v}: {len(cmp)} fotos · IoU mediana {iou.median():.2f} · "
              f"IoU >= 0,5 en {(iou >= 0.5).mean():.0%} · IoU < 0,3: {int((iou < 0.3).sum())} fotos")

## 3. Qué mide cada métrica

Mirando una clase (por ejemplo **dorsal**): VP = anotado dorsal y el modelo dice dorsal; FP = el modelo dice dorsal pero no lo es; FN = anotado dorsal y el modelo dice otra cosa o `unclear`.

- **Exactitud (accuracy)** = acertadas / todas (un `unclear` cuenta como error). **accuracy_when_decided**: solo donde el modelo decidió. **Cobertura**: fracción donde decidió.
- **Precisión** = VP / (VP + FP): cuando dice dorsal, ¿cuántas veces lo es?
- **Recall** = VP / (VP + FN): de los dorsales, ¿cuántos encontró?
- **F1** = 2·P·R / (P + R): alta solo si las dos lo son.
- **macro F1** y **exactitud balanceada**: cada clase pesa lo mismo.
- **Kappa de Cohen** (pendiente de agregar): acuerdo descontando el azar; para la erosión, kappa ponderado.

## 4. Persona vs modelos, con veredicto

Todas las fotos anotadas de A1, A2 y B (filtro *Run*). Se usan los prompts presentes en las tres corridas; la segunda celda revisa **B con v15**. Para cada atributo con desacuerdo, marcar **My error**, **Model error** o **Unsure / ask Martin**, y *Save* (la foto queda revisada). Los veredictos se guardan en `reviews/` de cada corrida; no cambian ninguna anotación.

In [ ]:
COMMON = [v for v in VERSIONS if all(v in available(c) for c in RUNS)]
print('Prompts en las 3 corridas:', COMMON)
DisagreementApp(RUNS, COMMON, ANNOTATOR, view='any_model_differs',
                mode_of=TOP3 if all(v in COMMON for v in TOP3) else None).show()

In [ ]:
# Solo B, con v15
DisagreementApp([cfg_b], available(cfg_b), ANNOTATOR, view='any_model_differs').show()

In [ ]:
# Resumen de veredictos: registros a corregir en el notebook 02 y preguntas para Martin
for c in RUNS:
    print('=====', c.base_dir.split('/')[-1])
    verdict_summary(c, ANNOTATOR)

## 5. Registros posiblemente duplicados

Pares de registros del **mismo día** y a ≤ 2 km (primero se filtra por día, después se calcula la distancia). Solo son candidatos: si es el mismo hueso, se marca en la anotación (*Same bone as*).

In [ ]:
from sepia.duplicates import load_runs, candidate_pairs, candidate_groups
obs_all = load_runs(cfg_a1, ['pilot_a1', 'a2_test', 'sector_b'])
pairs = candidate_pairs(obs_all, max_km=2)
groups = candidate_groups(pairs)
print(f"{len(pairs)} pares · {len(groups)} grupos")
display(groups)
display(pairs)